In [106]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression, ElasticNet, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [79]:
df = pd.read_csv("data/processed/data.csv")

In [80]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21771 entries, 0 to 21770
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   trip_distance  21771 non-null  float64
 1   RatecodeID     21771 non-null  int64  
 2   fare_amount    21771 non-null  float64
 3   trip_duration  21771 non-null  float64
 4   pickup_hr      21771 non-null  int64  
dtypes: float64(3), int64(2)
memory usage: 850.6 KB


In [81]:
df.head()

,trip_distance,RatecodeID,fare_amount,trip_duration,pickup_hr
0,3.34,1,13.0,0.234444,8
1,1.80,1,16.0,0.441667,14
2,1.00,1,6.5,0.120000,7
3,3.70,1,20.5,0.504167,13
4,4.37,1,16.5,0.278611,23


In [82]:
df.loc[:, "is_early_morning"] = np.where(df["pickup_hr"].isin([2, 3, 4]), 1, 0)

In [83]:
df.groupby("is_early_morning")["fare_amount"].agg(["median", "mean"])

,median,mean
is_early_morning,,
0,9.0,11.739412
1,10.0,12.542334


In [84]:
df.loc[:, "is_morning"] = np.where(df["pickup_hr"].isin([5, 6, 7]), 1, 0)

In [85]:
df.groupby("is_morning")["fare_amount"].agg(["median", "mean"])

,median,mean
is_morning,,
0,9.5,11.845945
1,8.0,10.766845


In [86]:
df = df.drop("pickup_hr", axis=1)

In [87]:
df

,trip_distance,RatecodeID,fare_amount,trip_duration,is_early_morning,is_morning
0,3.34,1,13.0,0.234444,0,0
1,1.80,1,16.0,0.441667,0,0
2,1.00,1,6.5,0.120000,0,1
3,3.70,1,20.5,0.504167,0,0
4,4.37,1,16.5,0.278611,0,0
...,...,...,...,...,...,...
21766,0.89,1,7.5,0.157500,0,0
21767,0.61,1,4.0,0.054444,0,0
21768,0.42,1,4.5,0.068889,0,0
21769,2.36,1,10.5,0.198889,0,0


# Data Split

In [88]:
X = df.drop(["fare_amount", "trip_duration"], axis=1)
y = df["fare_amount"]

In [89]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [90]:
X_train

,trip_distance,RatecodeID,is_early_morning,is_morning
11266,8.72,1,0,1
4411,0.51,1,0,0
17031,0.73,1,0,0
21706,1.64,1,0,0
3051,0.88,1,0,0
...,...,...,...,...
13123,2.47,1,0,0
19648,0.50,1,0,0
9845,2.13,1,0,0
10799,0.40,1,0,1


In [91]:
scaler = StandardScaler()

# Fit on training data and transform both sets
X_train["trip_distance"] = scaler.fit_transform(X_train[["trip_distance"]])
X_test["trip_distance"] = scaler.transform(X_test[["trip_distance"]])

# Model training

Linear Regression

In [92]:
lr = LinearRegression()
lr.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [93]:
yh = lr.predict(X_test)

In [94]:
def evaluate_model(model, name, y_test):
    yh = model.predict(X_test)
    mae = mean_absolute_error(y_test, yh)
    mse = mean_squared_error(y_test, yh)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, yh)
    n = len(y_test)
    k = 1
    adjusted_r2 = 1 - ((1 - r2) * (n - 1) / (n - k - 1))

    table = pd.DataFrame(
        {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2, "Adjusted R2": adjusted_r2},
        index=[name],
    )
    return table


In [95]:
lr_result = evaluate_model(lr, "linear regression", y_test)
lr_result

,MAE,MSE,RMSE,R2,Adjusted R2
linear regression,1.750004,6.413203,2.53243,0.899717,0.899694


In [96]:
# Check which features your model is relying on so heavily
import pandas as pd

# If using scikit-learn LinearRegression:
coefficients = pd.DataFrame(
    {"Feature": X_train.columns, "Coefficient": lr.coef_}
).sort_values(by="Coefficient", ascending=False)

print(coefficients)

            Feature  Coefficient
0     trip_distance     7.690441
1        RatecodeID     3.923621
3        is_morning    -1.321358
2  is_early_morning    -1.600278


---

ElasticNet

In [97]:
base_model = ElasticNet(max_iter=2000)

param_grid = {"alpha": [0.01, 0.1, 1.0, 10.0], "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]}

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=5,
)

# 5. Run the search
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
best_params = grid_search.best_params_

In [98]:
en = ElasticNet(**best_params)
en.fit(X_train, y_train)

,alpha,0.01
,l1_ratio,0.9
,fit_intercept,True
,precompute,False
,max_iter,1000
,copy_X,True
,tol,0.0001
,warm_start,False
,positive,False
,random_state,None
,selection,'cyclic'


In [99]:
table = pd.concat([lr_result, evaluate_model(best_model, "ElasticNet", y_test)])

In [100]:
table

,MAE,MSE,RMSE,R2,Adjusted R2
linear regression,1.750004,6.413203,2.532430,0.899717,0.899694
ElasticNet,1.753926,6.514563,2.552364,0.898132,0.898109


---

Ridge


In [ ]:
ridge_base = Ridge()

# Testing a wide log-space range of alpha values (from tiny to large penalties)
param_grid = {"alpha": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}

grid_search_ridge = GridSearchCV(
    estimator=ridge_base, param_grid=param_grid, cv=5, scoring="r2", n_jobs=-1
)

grid_search_ridge.fit(X_train, y_train)

best_ridge_model = grid_search_ridge.best_estimator_

ridge_grid_results = evaluate_model(
    best_ridge_model,
    f"Ridge (Opt Alpha={grid_search_ridge.best_params_['alpha']})",
    y_test,
)


In [102]:
table = pd.concat([table, ridge_grid_results])

In [103]:
table


,MAE,MSE,RMSE,R2,Adjusted R2
linear regression,1.750004,6.413203,2.532430,0.899717,0.899694
ElasticNet,1.753926,6.514563,2.552364,0.898132,0.898109
Ridge (Opt Alpha=10.0),1.750474,6.424928,2.534744,0.899534,0.899511


Model selected: Linear Regression

In [105]:
import pickle

with open("models/model.pkl", "wb") as f:
    pickle.dump(en, f)